In [1]:
# DiffDock Batch Docking Pipeline - Simplified Version
#
# This notebook docks all protein-ligand combinations using DiffDock.
# Uses the standard DiffDock inference with CSV input for batch processing.
#
# Reference: https://github.com/gcorso/DiffDock
#
# Usage:
#   python -m inference --config default_inference_args.yaml \
#     --protein_ligand_csv input.csv --out_dir ./results
#
# For single complex:
#   python -m inference --protein_path protein.pdb --ligand ligand.sdf --out_dir ./output

In [2]:
from __future__ import annotations

import csv
import json
import os
import shutil
import subprocess
import time
from dataclasses import dataclass, field
from datetime import datetime
from pathlib import Path
from typing import Dict, List, Optional, Tuple
import numpy as np
from collections import defaultdict

In [3]:
# ============================================================================
# CONFIGURATION - Adjust these parameters as needed
# ============================================================================

# Paths
workspace_root = Path("/home/manndo/master_dev")
drugs_dir = workspace_root / "docking_ready_diffdock/Drugs"  # Directory containing ligand files for DiffDock
receptors_dir = workspace_root / "Orai"

# DiffDock paths
DIFFDOCK_DIR = Path("/home/manndo/docking_tools/DiffDock")
DIFFDOCK_CONDA_ENV = "diffdock"

# Number of samples (poses) DiffDock generates per complex
NUM_SAMPLES: int = 30

# Device for DiffDock (cpu or cuda:0)
DIFFDOCK_DEVICE = "cuda:0"

# Output directories
DIFFDOCK_OUTPUT_DIR = workspace_root / "diffdock_results"
DIFFDOCK_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Overwrite settings
OVERWRITE_EXISTING = True  # Set to True to re-dock existing combinations

# Validate directories
if not drugs_dir.exists():
    raise FileNotFoundError(f"Ligand directory missing: {drugs_dir}")
if not receptors_dir.exists():
    raise FileNotFoundError(f"Receptor directory missing: {receptors_dir}")

print("=" * 80)
print("DiffDock Batch Docking Configuration")
print("=" * 80)
print(f"Number of samples per complex: {NUM_SAMPLES}")
print(f"DiffDock directory: {DIFFDOCK_DIR}")
print(f"DiffDock conda env: {DIFFDOCK_CONDA_ENV}")
print(f"DiffDock device: {DIFFDOCK_DEVICE}")
print(f"Output directory: {DIFFDOCK_OUTPUT_DIR}")
print(f"Overwrite existing: {OVERWRITE_EXISTING}")
print()

# Validate DiffDock installation
if not DIFFDOCK_DIR.exists():
    print(f"⚠️  WARNING: DiffDock directory not found at {DIFFDOCK_DIR}")
    print("   Please update DIFFDOCK_DIR to point to your DiffDock installation")
else:
    print(f"✓ DiffDock directory found")

DiffDock Batch Docking Configuration
Number of samples per complex: 30
DiffDock directory: /home/manndo/docking_tools/DiffDock
DiffDock conda env: diffdock
DiffDock device: cuda:0
Output directory: /home/manndo/master_dev/diffdock_results
Overwrite existing: True

✓ DiffDock directory found


In [4]:

# ============================================================================
# HELPER FUNCTIONS
# ============================================================================

@dataclass
class DockingResult:
    """Result of docking a protein-ligand combination."""
    protein_name: str
    ligand_name: str
    protein_path: Path
    ligand_path: Path
    output_dir: Path
    status: str  # "success", "failed", "skipped"
    num_poses: int = 0
    pose_files: List[Path] = field(default_factory=list)
    error_message: str = ""
    elapsed_time: float = 0.0
    
    def to_dict(self) -> dict:
        return {
            "protein_name": self.protein_name,
            "ligand_name": self.ligand_name,
            "protein_path": str(self.protein_path),
            "ligand_path": str(self.ligand_path),
            "output_dir": str(self.output_dir),
            "status": self.status,
            "num_poses": self.num_poses,
            "pose_files": [str(p) for p in self.pose_files],
            "error_message": self.error_message,
            "elapsed_time": self.elapsed_time,
        }


def get_file_stem(path: Path) -> str:
    """Get clean filename stem without extension."""
    return path.stem.replace("_ligand", "").replace("_protein", "")


def collect_files(root: Path, extensions: List[str]) -> List[Path]:
    """Return files with given extensions located directly inside root (no recursion)."""
    files = []
    for ext in extensions:
        files.extend(sorted(p for p in root.glob(f"*{ext}") if p.is_file()))
    return sorted(set(files))


def extract_confidence_from_filename(filename: str) -> float:
    """Extract confidence score from DiffDock output filename (e.g., rank1_confidence-0.85.sdf)."""
    try:
        if "confidence" in filename.lower():
            parts = filename.lower().split("confidence")
            if len(parts) > 1:
                conf_str = parts[1].replace("-", "").replace("_", "").replace(".sdf", "")
                return float(conf_str)
    except:
        pass
    return 0.0


def extract_rank_from_filename(filename: str) -> int:
    """Extract rank from DiffDock output filename (e.g., rank1_confidence-0.85.sdf)."""
    try:
        if "rank" in filename.lower():
            parts = filename.lower().split("rank")
            if len(parts) > 1:
                rank_str = ""
                for c in parts[1]:
                    if c.isdigit():
                        rank_str += c
                    else:
                        break
                if rank_str:
                    return int(rank_str)
    except:
        pass
    return 0


def prepare_protein_for_diffdock(input_pdb: Path, output_dir: Path) -> Path:
    """
    Prepare a protein PDB file for DiffDock by fixing common issues:
    - Convert non-standard histidine names (HSD, HSE, HSP) to standard HIS
    - Remove HETATM records (water, ions, ligands)
    - Keep only ATOM records
    
    Accepts a single PDB file or a directory of PDB files.
    If a directory is given, all .pdb files inside are prepared.
    
    Args:
        input_pdb: Path to input PDB file or directory of PDB files
        output_dir: Directory to save prepared PDB(s)
    
    Returns:
        Path to prepared PDB file, or output_dir if a directory was given
    """
    if input_pdb.is_dir():
        pdb_files = sorted(input_pdb.glob("*.pdb"))
        if not pdb_files:
            raise FileNotFoundError(f"No .pdb files found in directory: {input_pdb}")
        print(f"Preparing {len(pdb_files)} PDB files from {input_pdb.name}/")
        for pdb in pdb_files:
            prepared = _prepare_single_protein(pdb, output_dir)
            print(f"  ✓ {prepared.name}")
        return output_dir
    else:
        return _prepare_single_protein(input_pdb, output_dir)


def _prepare_single_protein(input_pdb: Path, output_dir: Path) -> Path:
    """Prepare a single PDB file for DiffDock."""
    output_dir.mkdir(parents=True, exist_ok=True)
    output_pdb = output_dir / f"{input_pdb.stem}_prepared.pdb"
    
    # Residue name mapping (CHARMM/NAMD histidine variants -> standard)
    residue_mapping = {
        'HSD': 'HIS',  # delta-protonated histidine
        'HSE': 'HIS',  # epsilon-protonated histidine  
        'HSP': 'HIS',  # doubly protonated histidine
        'HIE': 'HIS',  # AMBER epsilon-protonated
        'HID': 'HIS',  # AMBER delta-protonated
        'HIP': 'HIS',  # AMBER doubly protonated
    }
    
    with open(input_pdb, 'r') as f_in, open(output_pdb, 'w') as f_out:
        for line in f_in:
            # Only keep ATOM records (skip HETATM, waters, etc.)
            if line.startswith('ATOM'):
                # Fix non-standard residue names
                res_name = line[17:20].strip()
                if res_name in residue_mapping:
                    # Replace residue name in the line (columns 18-20)
                    new_res = residue_mapping[res_name]
                    line = line[:17] + f"{new_res:>3}" + line[20:]
                f_out.write(line)
            elif line.startswith(('END', 'TER')):
                f_out.write(line)
    
    return output_pdb


def prepare_ligand_for_diffdock(input_sdf: Path, output_dir: Path) -> Path:
    """
    Prepare a ligand SDF file for DiffDock by fixing valence/charge issues.
    
    Common problem: protonated atoms (e.g., NH3+) have 4 bonds in the SDF but
    no M  CHG record, causing RDKit AtomValenceException.
    
    This function:
    1. Reads the V2000 atom block for explicit valence hints
    2. Adds missing M  CHG records for atoms whose bond count exceeds
       the default valence (e.g., N with 4 bonds -> charge +1)
    3. Writes a corrected SDF file
    
    Args:
        input_sdf: Path to input SDF file
        output_dir: Directory to save prepared SDF
    
    Returns:
        Path to prepared SDF file (original returned if no fix needed)
    """
    output_dir.mkdir(parents=True, exist_ok=True)
    output_sdf = output_dir / f"{input_sdf.stem}_prepared.sdf"
    
    # Default valences for common elements
    default_valence = {
        'C': 4, 'N': 3, 'O': 2, 'S': 2, 'P': 3,
        'F': 1, 'Cl': 1, 'Br': 1, 'I': 1, 'B': 3,
    }
    
    with open(input_sdf, 'r') as f:
        lines = f.readlines()
    
    # Parse counts line (line 4, 0-indexed line 3)
    if len(lines) < 5:
        return input_sdf  # Not a valid SDF
    
    counts_line = lines[3]
    try:
        num_atoms = int(counts_line[:3])
        num_bonds = int(counts_line[3:6])
    except ValueError:
        return input_sdf
    
    # Parse atom block to find explicit valence hints
    # V2000 atom line: x y z symbol massdiff charge stereo hcount stereocare valence ...
    atom_symbols = []
    explicit_valences = []  # from the valence column (column index 48-50 in fixed-width)
    
    for i in range(4, 4 + num_atoms):
        line = lines[i]
        parts = line.split()
        if len(parts) >= 10:
            symbol = parts[3]
            atom_symbols.append(symbol)
            # The 10th field (index 9) is the valence column in some SDF writers,
            # but in standard V2000 it's at fixed position. Let's use the field approach:
            # fields: x, y, z, symbol, massdiff, charge, stereo, hcount, stereocare, valence
            try:
                val = int(parts[9])
            except (ValueError, IndexError):
                val = 0
            explicit_valences.append(val)
        else:
            atom_symbols.append('?')
            explicit_valences.append(0)
    
    # Count bonds per atom
    bond_counts = [0] * num_atoms
    for i in range(4 + num_atoms, 4 + num_atoms + num_bonds):
        line = lines[i]
        parts = line.split()
        if len(parts) >= 2:
            a1 = int(parts[0]) - 1  # 1-indexed to 0-indexed
            a2 = int(parts[1]) - 1
            bond_order = int(parts[2]) if len(parts) >= 3 else 1
            bond_counts[a1] += bond_order
            bond_counts[a2] += bond_order
    
    # Find atoms that need formal charges
    charges_to_add = {}  # atom_index (1-indexed) -> charge
    for idx in range(num_atoms):
        symbol = atom_symbols[idx]
        if symbol not in default_valence:
            continue
        dv = default_valence[symbol]
        bc = bond_counts[idx]
        ev = explicit_valences[idx]
        
        # If bond count exceeds default valence, atom likely needs a positive charge
        if bc > dv and ev > 0:
            charge = bc - dv
            charges_to_add[idx + 1] = charge  # 1-indexed
    
    if not charges_to_add:
        # No fixes needed, return original
        return input_sdf
    
    # Insert M  CHG line(s) before M  END
    charge_atoms = list(charges_to_add.items())
    chg_line = f"M  CHG  {len(charge_atoms)}"
    for atom_idx, charge in charge_atoms:
        chg_line += f"  {atom_idx:3d}  {charge:3d}"
    chg_line += "\n"
    
    # Find M  END and insert before it
    with open(output_sdf, 'w') as f:
        for line in lines:
            if line.strip() == "M  END":
                f.write(chg_line)
            f.write(line)
    
    fixed_atoms = ", ".join(f"atom #{k} ({atom_symbols[k-1]}) -> +{v}" for k, v in charges_to_add.items())
    print(f"  ⚠ Fixed missing charges in {input_sdf.name}: {fixed_atoms}")
    
    return output_sdf


print("Helper functions defined.")


Helper functions defined.


In [5]:


# ============================================================================
# DIFFDOCK INFERENCE FUNCTIONS
# ============================================================================

def create_protein_ligand_csv(
    proteins: List[Path],
    ligands: List[Path],
    output_csv: Path,
    prepare: bool = True,
) -> List[Tuple[str, Path, Path]]:
    """
    Create a CSV file for DiffDock batch inference.
    
    If prepare=True, each protein is run through prepare_protein_for_diffdock
    (fix HSD/HSE/HSP -> HIS, strip HETATM) and each SDF ligand through
    prepare_ligand_for_diffdock (fix missing formal charges) before writing.
    
    CSV format: complex_name, protein_path, ligand_description, protein_sequence
    """
    combinations = []
    
    # Optionally prepare proteins and ligands once (cache to avoid re-preparing)
    prepared_protein_cache: Dict[Path, Path] = {}
    prepared_ligand_cache: Dict[Path, Path] = {}
    if prepare:
        prepared_prot_dir = output_csv.parent / "prepared_proteins"
        for protein in proteins:
            if protein not in prepared_protein_cache:
                prepared_protein_cache[protein] = prepare_protein_for_diffdock(protein, prepared_prot_dir)
                print(f"  Prepared protein: {prepared_protein_cache[protein].name}")
        
        prepared_lig_dir = output_csv.parent / "prepared_ligands"
        for ligand in ligands:
            if ligand not in prepared_ligand_cache:
                if ligand.suffix.lower() == ".sdf":
                    prepared_ligand_cache[ligand] = prepare_ligand_for_diffdock(ligand, prepared_lig_dir)
                else:
                    prepared_ligand_cache[ligand] = ligand
    
    with open(output_csv, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['complex_name', 'protein_path', 'ligand_description', 'protein_sequence'])
        
        for protein in proteins:
            protein_name = get_file_stem(protein)
            protein_to_use = prepared_protein_cache.get(protein, protein)
            for ligand in ligands:
                ligand_name = get_file_stem(ligand)
                ligand_to_use = prepared_ligand_cache.get(ligand, ligand)
                complex_name = f"{ligand_name}__{protein_name}"
                
                writer.writerow([
                    complex_name,
                    str(protein_to_use.absolute()),
                    str(ligand_to_use.absolute()),
                    ''  # Empty protein_sequence since we have PDB files
                ])
                combinations.append((complex_name, protein, ligand))
    
    print(f"Created CSV with {len(combinations)} combinations: {output_csv}")
    return combinations


def run_diffdock_batch(
    csv_path: Path,
    output_dir: Path,
    samples: int = NUM_SAMPLES,
    device: str = DIFFDOCK_DEVICE,
) -> Tuple[bool, str]:
    """
    Run DiffDock inference using CSV input for batch processing.
    Proteins and ligands should already be prepared via create_protein_ligand_csv(prepare=True).
    """
    output_dir.mkdir(parents=True, exist_ok=True)
    
    import sys
    cmd = [
        sys.executable, "-m", "inference",
        "--config", "default_inference_args.yaml",
        "--protein_ligand_csv", str(csv_path),
        "--out_dir", str(output_dir),
        "--samples_per_complex", str(samples),
        "--no_final_step_noise",
        "--save_visualisation",
    ]
    
    # Control GPU vs CPU via CUDA_VISIBLE_DEVICES
    env = os.environ.copy()
    if device == "cpu":
        env["CUDA_VISIBLE_DEVICES"] = ""
    
    print(f"Running DiffDock batch inference (device={device})...")
    print(f"  Command: {' '.join(cmd)}")
    print(f"  Working dir: {DIFFDOCK_DIR}")
    
    try:
        result = subprocess.run(
            cmd,
            capture_output=True,
            text=True,
            timeout=7200,
            cwd=str(DIFFDOCK_DIR),
            env=env,
        )
        
        if result.returncode == 0:
            return True, ""
        else:
            error_msg = f"Return code {result.returncode}. stderr: {result.stderr[-1000:] if result.stderr else 'None'}"
            return False, error_msg
            
    except subprocess.TimeoutExpired:
        return False, "Batch docking timeout (7200s)"
    except Exception as e:
        return False, str(e)


def _run_diffdock_subprocess(
    protein_path: Path,
    ligand_path: Path,
    output_dir: Path,
    samples: int,
    device: str,
) -> Tuple[subprocess.CompletedProcess, List[Path]]:
    """
    Internal helper: run DiffDock subprocess and collect output SDF files.
    Uses the same Python interpreter as the current notebook kernel.
    """
    import sys
    cmd = [
        sys.executable, "-m", "inference",
        "--config", "default_inference_args.yaml",
        "--protein_path", str(protein_path),
        "--ligand_description", str(ligand_path),
        "--out_dir", str(output_dir),
        "--samples_per_complex", str(samples),
        "--no_final_step_noise",
        "--save_visualisation",
    ]
    
    # Control GPU vs CPU via CUDA_VISIBLE_DEVICES
    env = os.environ.copy()
    if device == "cpu":
        env["CUDA_VISIBLE_DEVICES"] = ""
    
    result = subprocess.run(
        cmd,
        capture_output=True,
        text=True,
        timeout=7200,
        cwd=str(DIFFDOCK_DIR),
        env=env,
    )
    
    # Collect output SDF files
    output_sdfs = []
    for sdf in output_dir.rglob("*.sdf"):
        if sdf.stat().st_size > 0:
            output_sdfs.append(sdf)
    output_sdfs = sorted(output_sdfs, key=lambda p: extract_rank_from_filename(p.name))
    
    return result, output_sdfs


def run_diffdock_single(
    protein_path: Path,
    ligand_path: Path,
    output_dir: Path,
    samples: int = NUM_SAMPLES,
    device: str = DIFFDOCK_DEVICE,
) -> Tuple[bool, List[Path], str]:
    """
    Run DiffDock inference for a single protein-ligand pair.
    
    If device is 'cuda:0' or 'cuda', tries GPU first.
    On CUDA OOM, automatically retries on CPU.
    """
    output_dir.mkdir(parents=True, exist_ok=True)
    
    try:
        result, output_sdfs = _run_diffdock_subprocess(
            protein_path, ligand_path, output_dir, samples, device,
        )
        
        if output_sdfs:
            return True, output_sdfs, ""
        
        # Check if failure was CUDA OOM — retry on CPU
        stderr = result.stderr or ""
        if "OutOfMemoryError" in stderr or "CUDA out of memory" in stderr:
            if device != "cpu":
                print(f"  ⚠ CUDA OOM — retrying on CPU...")
                # Clean up any partial output
                if output_dir.exists():
                    shutil.rmtree(output_dir)
                output_dir.mkdir(parents=True, exist_ok=True)
                
                result, output_sdfs = _run_diffdock_subprocess(
                    protein_path, ligand_path, output_dir, samples, "cpu",
                )
                
                if output_sdfs:
                    return True, output_sdfs, ""
                
                error_msg = f"CPU fallback also failed. stderr: {result.stderr[-500:] if result.stderr else ''}"
                return False, [], error_msg
        
        error_msg = f"No output SDF files found. stderr: {stderr[-500:]}"
        return False, [], error_msg
        
    except subprocess.TimeoutExpired:
        return False, [], "Docking timeout (7200s)"
    except Exception as e:
        return False, [], str(e)


print("DiffDock inference functions defined.")



DiffDock inference functions defined.


In [6]:

# ============================================================================
# BATCH DOCKING FUNCTIONS
# ============================================================================

def dock_single_combination(
    protein_path: Path,
    ligand_path: Path,
    output_base_dir: Path,
    samples: int = NUM_SAMPLES,
    device: str = DIFFDOCK_DEVICE,
) -> DockingResult:
    """Dock a single protein-ligand combination."""
    protein_name = get_file_stem(protein_path)
    ligand_name = get_file_stem(ligand_path)
    combo_name = f"{ligand_name}__{protein_name}"
    output_dir = output_base_dir / combo_name
    
    result = DockingResult(
        protein_name=protein_name,
        ligand_name=ligand_name,
        protein_path=protein_path,
        ligand_path=ligand_path,
        output_dir=output_dir,
        status="pending",
    )
    
    # Check if already exists
    if not OVERWRITE_EXISTING and output_dir.exists():
        existing_sdfs = list(output_dir.rglob("*.sdf"))
        if existing_sdfs:
            result.status = "skipped"
            result.num_poses = len(existing_sdfs)
            result.pose_files = sorted(existing_sdfs, key=lambda p: extract_rank_from_filename(p.name))
            return result
    
    start_time = time.time()
    
    # Prepare protein (fix non-standard residue names like HSD/HSE/HSP -> HIS)
    prepared_dir = output_base_dir / "prepared_proteins"
    prepared_protein = prepare_protein_for_diffdock(protein_path, prepared_dir)
    print(f"  Prepared protein: {prepared_protein.name}")
    
    # Prepare ligand (fix missing formal charges for protonated atoms)
    prepared_ligand_dir = output_base_dir / "prepared_ligands"
    if ligand_path.suffix.lower() == ".sdf":
        prepared_ligand = prepare_ligand_for_diffdock(ligand_path, prepared_ligand_dir)
    else:
        prepared_ligand = ligand_path
    
    success, pose_files, error = run_diffdock_single(
        protein_path=prepared_protein,
        ligand_path=prepared_ligand,
        output_dir=output_dir,
        samples=samples,
        device=device,
    )
    
    result.elapsed_time = time.time() - start_time
    
    if success and pose_files:
        result.status = "success"
        result.num_poses = len(pose_files)
        result.pose_files = pose_files
    else:
        result.status = "failed"
        result.error_message = error
    
    return result


def run_diffdock_sequential(
    proteins: List[Path],
    ligands: List[Path],
    output_dir: Path,
    samples: int = NUM_SAMPLES,
    device: str = DIFFDOCK_DEVICE,
) -> List[DockingResult]:
    """
    Run DiffDock for all protein-ligand combinations sequentially.
    Each combination is docked one at a time.
    """
    total = len(proteins) * len(ligands)
    results = []
    
    print("=" * 80)
    print("DiffDock Sequential Docking")
    print("=" * 80)
    print(f"Proteins: {len(proteins)}")
    print(f"Ligands: {len(ligands)}")
    print(f"Total combinations: {total}")
    print(f"Samples per complex: {samples}")
    print()
    
    idx = 0
    for protein in proteins:
        for ligand in ligands:
            idx += 1
            print(f"\n[{idx}/{total}] {get_file_stem(ligand)} + {get_file_stem(protein)}")
            
            result = dock_single_combination(
                protein_path=protein,
                ligand_path=ligand,
                output_base_dir=output_dir,
                samples=samples,
                device=device,
            )
            
            results.append(result)
            
            status_icon = {"success": "✓", "failed": "✗", "skipped": "⊘"}.get(result.status, "?")
            print(f"  {status_icon} {result.status} | Poses: {result.num_poses} | Time: {result.elapsed_time:.1f}s")
            
            if result.error_message:
                print(f"    Error: {result.error_message[:100]}")
    
    return results


print("Batch docking functions defined.")


Batch docking functions defined.


In [7]:
# ============================================================================
# SUMMARY FUNCTIONS
# ============================================================================

def generate_summary(results: List[DockingResult]) -> Dict:
    """Generate summary of docking results."""
    summary = {
        "timestamp": datetime.now().isoformat(),
        "configuration": {
            "num_samples": NUM_SAMPLES,
            "device": DIFFDOCK_DEVICE,
            "diffdock_dir": str(DIFFDOCK_DIR),
        },
        "overall": {
            "total_combinations": len(results),
            "successful": sum(1 for r in results if r.status == "success"),
            "failed": sum(1 for r in results if r.status == "failed"),
            "skipped": sum(1 for r in results if r.status == "skipped"),
            "total_poses": sum(r.num_poses for r in results),
            "total_time_seconds": sum(r.elapsed_time for r in results),
        },
        "by_protein": defaultdict(lambda: {"combinations": 0, "poses": 0, "success": 0}),
        "by_ligand": defaultdict(lambda: {"combinations": 0, "poses": 0, "success": 0}),
        "combinations": [],
    }
    
    for r in results:
        summary["by_protein"][r.protein_name]["combinations"] += 1
        summary["by_protein"][r.protein_name]["poses"] += r.num_poses
        if r.status == "success":
            summary["by_protein"][r.protein_name]["success"] += 1
            
        summary["by_ligand"][r.ligand_name]["combinations"] += 1
        summary["by_ligand"][r.ligand_name]["poses"] += r.num_poses
        if r.status == "success":
            summary["by_ligand"][r.ligand_name]["success"] += 1
        
        summary["combinations"].append({
            "protein": r.protein_name,
            "ligand": r.ligand_name,
            "status": r.status,
            "num_poses": r.num_poses,
            "time_seconds": round(r.elapsed_time, 2),
            "error": r.error_message if r.error_message else None,
        })
    
    summary["by_protein"] = dict(summary["by_protein"])
    summary["by_ligand"] = dict(summary["by_ligand"])
    
    return summary


def print_summary(summary: Dict):
    """Print formatted docking summary."""
    print("\n" + "=" * 80)
    print("DIFFDOCK DOCKING SUMMARY")
    print("=" * 80)
    
    overall = summary["overall"]
    print(f"\nOverall Statistics:")
    print(f"  Total combinations: {overall['total_combinations']}")
    print(f"  Successful: {overall['successful']}")
    print(f"  Failed: {overall['failed']}")
    print(f"  Skipped: {overall['skipped']}")
    print(f"  Total poses generated: {overall['total_poses']}")
    print(f"  Total time: {overall['total_time_seconds']:.1f}s ({overall['total_time_seconds']/60:.1f} min)")
    
    print("\n" + "-" * 80)
    print("Results by Protein:")
    print("-" * 80)
    for protein, stats in sorted(summary["by_protein"].items()):
        print(f"  {protein:40s} | Combos: {stats['combinations']:3d} | Poses: {stats['poses']:4d}")
    
    print("\n" + "-" * 80)
    print("Results by Ligand:")
    print("-" * 80)
    for ligand, stats in sorted(summary["by_ligand"].items()):
        print(f"  {ligand:40s} | Combos: {stats['combinations']:3d} | Poses: {stats['poses']:4d}")


print("Summary functions defined.")

Summary functions defined.


In [8]:

# ============================================================================
# COLLECT INPUT FILES
# ============================================================================

# Ligands: SDF, MOL2, PDB files from drugs_dir
# Proteins: PDB files from receptors_dir

ligand_files = collect_files(drugs_dir, [".sdf", ".mol2", ".pdb"])
all_receptor_files = collect_files(receptors_dir, [".pdb"])

# Filter to only keep receptor files that contain a frame number (e.g., Fr300, Fr0)
# Handles both raw names (ending in digit) and cleaned names (ending in _cleaned)
import re
receptor_files = [f for f in all_receptor_files if re.search(r'Fr\d+', f.stem)]
print(f"Filtered receptors: {len(receptor_files)} of {len(all_receptor_files)} (keeping only files with frame numbers)")

print("=" * 80)
print("INPUT FILES FOR DIFFDOCK")
print("=" * 80)

print(f"\nProteins ({len(receptor_files)} files from {receptors_dir}):")
for pdb in receptor_files:
    print(f"  - {pdb.name}")

print(f"\nLigands ({len(ligand_files)} files from {drugs_dir}):")
for lig in ligand_files:
    print(f"  - {lig.name}")

print(f"\nTotal docking combinations: {len(receptor_files) * len(ligand_files)}")
print(f"Samples per complex: {NUM_SAMPLES}")
print(f"Expected total poses: {len(receptor_files) * len(ligand_files) * NUM_SAMPLES}")


Filtered receptors: 4 of 4 (keeping only files with frame numbers)
INPUT FILES FOR DIFFDOCK

Proteins (4 files from /home/manndo/master_dev/Orai):
  - Orai1WT-MDSnap-Fr300_cleaned.pdb
  - Orai1WT-MDSnap-Fr400_cleaned.pdb
  - Orai1WT-MDSnap-Fr499_cleaned.pdb
  - Orai1WT-START-Fr0_cleaned.pdb

Ligands (5 files from /home/manndo/master_dev/docking_ready_diffdock/Drugs):
  - 2abp-nh2-OPT.sdf
  - 2abp-nh3p-OPT.sdf
  - Synta-66-OPT-Singlet.sdf
  - gsk7975a-deprot-OPT.sdf
  - gsk7975a-prot-OPT.sdf

Total docking combinations: 20
Samples per complex: 30
Expected total poses: 600


In [ ]:
# ============================================================================
# RUN DIFFDOCK DOCKING
# ============================================================================

# Run docking sequentially for each protein-ligand combination
diffdock_results = run_diffdock_sequential(
    proteins=receptor_files,
    ligands=ligand_files,
    output_dir=DIFFDOCK_OUTPUT_DIR,
    samples=NUM_SAMPLES,
    device=DIFFDOCK_DEVICE,
)

# Generate and display summary
docking_summary = generate_summary(diffdock_results)
print_summary(docking_summary)

# Save summary to file
summary_path = DIFFDOCK_OUTPUT_DIR / "docking_summary.json"
with open(summary_path, 'w') as f:
    json.dump(docking_summary, f, indent=2)
print(f"\nSummary saved to: {summary_path}")

DiffDock Sequential Docking
Proteins: 4
Ligands: 5
Total combinations: 20
Samples per complex: 30


[1/20] 2abp-nh2-OPT + Orai1WT-MDSnap-Fr300_cleaned
  Prepared protein: Orai1WT-MDSnap-Fr300_cleaned_prepared.pdb
  ✓ success | Poses: 91 | Time: 634.1s

[2/20] 2abp-nh3p-OPT + Orai1WT-MDSnap-Fr300_cleaned
  Prepared protein: Orai1WT-MDSnap-Fr300_cleaned_prepared.pdb
  ⚠ Fixed missing charges in 2abp-nh3p-OPT.sdf: atom #27 (N) -> +1
  ✗ failed | Poses: 0 | Time: 16.1s
    Error: No output SDF files found. stderr: eption: Explicit valence for atom # 26 N, 4, is greater than perm

[3/20] Synta-66-OPT-Singlet + Orai1WT-MDSnap-Fr300_cleaned
  Prepared protein: Orai1WT-MDSnap-Fr300_cleaned_prepared.pdb
  ✓ success | Poses: 61 | Time: 1329.8s

[4/20] gsk7975a-deprot-OPT + Orai1WT-MDSnap-Fr300_cleaned
  Prepared protein: Orai1WT-MDSnap-Fr300_cleaned_prepared.pdb
  ✓ success | Poses: 61 | Time: 1502.1s

[5/20] gsk7975a-prot-OPT + Orai1WT-MDSnap-Fr300_cleaned
  Prepared protein: Orai1WT-MDSnap-Fr3

In [ ]:
# ============================================================================
# VISUALIZE RESULTS
# ============================================================================
import pandas as pd

# Convert results to DataFrame
results_data = []
for r in diffdock_results:
    for i, pose_file in enumerate(r.pose_files):
        confidence = extract_confidence_from_filename(pose_file.name)
        rank = extract_rank_from_filename(pose_file.name)
        results_data.append({
            "protein": r.protein_name,
            "ligand": r.ligand_name,
            "rank": rank if rank > 0 else i + 1,
            "confidence": confidence,
            "sdf_path": str(pose_file),
            "status": r.status,
        })

results_df = pd.DataFrame(results_data)

print("=" * 80)
print("DOCKING RESULTS DATAFRAME")
print("=" * 80)
print(f"\nTotal poses: {len(results_df)}")

if not results_df.empty:
    print("\nPoses per protein-ligand combination:")
    pose_counts = results_df.groupby(["protein", "ligand"]).agg({
        "rank": "count",
        "confidence": "mean",
    }).reset_index()
    pose_counts.columns = ["protein", "ligand", "num_poses", "avg_confidence"]
    display(pose_counts)
    
    print("\nConfidence score distribution:")
    if results_df['confidence'].sum() > 0:
        print(f"  Mean confidence: {results_df['confidence'].mean():.3f}")
        print(f"  Max confidence: {results_df['confidence'].max():.3f}")
        print(f"  Min confidence: {results_df['confidence'].min():.3f}")

# Save to CSV
csv_path = DIFFDOCK_OUTPUT_DIR / "diffdock_poses.csv"
results_df.to_csv(csv_path, index=False)
print(f"\nResults saved to: {csv_path}")

DOCKING RESULTS DATAFRAME

Total poses: 526

Poses per protein-ligand combination:


,protein,ligand,num_poses,avg_confidence
0,Orai1WT-MDSnap-Fr300_cleaned,2abp-nh2-OPT,61,2.919344
1,Orai1WT-MDSnap-Fr300_cleaned,Synta-66-OPT-Singlet,31,3.080000
2,Orai1WT-MDSnap-Fr300_cleaned,gsk7975a-deprot-OPT,31,35.237742
3,Orai1WT-MDSnap-Fr300_cleaned,gsk7975a-prot-OPT,31,2.916452
4,Orai1WT-MDSnap-Fr400_cleaned,2abp-nh2-OPT,31,3.558065
5,Orai1WT-MDSnap-Fr400_cleaned,Synta-66-OPT-Singlet,31,2.942258
6,Orai1WT-MDSnap-Fr400_cleaned,gsk7975a-deprot-OPT,31,2.609355
7,Orai1WT-MDSnap-Fr400_cleaned,gsk7975a-prot-OPT,31,3.212258
8,Orai1WT-MDSnap-Fr499_cleaned,2abp-nh2-OPT,31,2.894839
9,Orai1WT-MDSnap-Fr499_cleaned,Synta-66-OPT-Singlet,31,3.096774



Confidence score distribution:
  Mean confidence: 6.798
  Max confidence: 1000.000
  Min confidence: 0.000

Results saved to: /home/manndo/master_dev/diffdock_results/diffdock_poses.csv


In [ ]:
# ============================================================================
# LIST ALL GENERATED POSES
# ============================================================================

def list_pose_files(output_dir: Path) -> Dict[str, List[Path]]:
    """List all generated pose files organized by combination."""
    poses_by_combo = {}
    
    if not output_dir.exists():
        return poses_by_combo
    
    for combo_dir in sorted(output_dir.iterdir()):
        if not combo_dir.is_dir():
            continue
        
        # DiffDock creates files like rank1_confidence-0.85.sdf
        pose_files = sorted(combo_dir.rglob("*.sdf"), key=lambda p: extract_rank_from_filename(p.name))
        if pose_files:
            poses_by_combo[combo_dir.name] = pose_files
    
    return poses_by_combo


pose_files = list_pose_files(DIFFDOCK_OUTPUT_DIR)

print("=" * 80)
print("GENERATED POSE FILES")
print("=" * 80)
print(f"\nOutput directory: {DIFFDOCK_OUTPUT_DIR}")
print(f"Total combinations with poses: {len(pose_files)}")
print()

total_poses = 0
for combo_name, files in pose_files.items():
    print(f"\n{combo_name}/")
    for f in files[:5]:  # Show first 5 poses per combination
        conf = extract_confidence_from_filename(f.name)
        print(f"  └── {f.name} (confidence: {conf:.3f})")
    if len(files) > 5:
        print(f"  └── ... and {len(files) - 5} more")
    total_poses += len(files)

print("\n" + "-" * 80)
print(f"TOTAL POSES GENERATED: {total_poses}")
print("-" * 80)

GENERATED POSE FILES

Output directory: /home/manndo/master_dev/diffdock_results
Total combinations with poses: 17


2abp-nh2-OPT__Orai1WT-MDSnap-Fr300_cleaned/
  └── rank1_confidence-0.62.sdf (confidence: 0.620)
  └── rank1.sdf (confidence: 0.000)
  └── rank1_confidence-0.65.sdf (confidence: 0.650)
  └── rank2_confidence-0.84.sdf (confidence: 0.840)
  └── rank2_confidence-0.65.sdf (confidence: 0.650)
  └── ... and 56 more

2abp-nh2-OPT__Orai1WT-MDSnap-Fr400_cleaned/
  └── rank1_confidence-0.56.sdf (confidence: 0.560)
  └── rank1.sdf (confidence: 0.000)
  └── rank2_confidence-1.00.sdf (confidence: 1.000)
  └── rank3_confidence-1.87.sdf (confidence: 1.870)
  └── rank4_confidence-2.03.sdf (confidence: 2.030)
  └── ... and 26 more

2abp-nh2-OPT__Orai1WT-MDSnap-Fr499_cleaned/
  └── rank1.sdf (confidence: 0.000)
  └── rank1_confidence-0.84.sdf (confidence: 0.840)
  └── rank2_confidence-1.13.sdf (confidence: 1.130)
  └── rank3_confidence-1.39.sdf (confidence: 1.390)
  └── rank4_confidence-1.